# LOBSTER bond indices 

Takes the `COBICAR.lobster` file of each material and integrates the COBI over the Wannier energy window.


In [1]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Inputs

`DIS_WIN_MIN` is from each `wannier90.win` (absolute DFT energy, eV). $N_e$ is the number of electrons in the Wannier manifold (Table S2).

In [2]:
BASE = 'data'

materials  = ['NaCl', 'Diamond', 'CsI3', 'GeTe']
dis_win_min = {'NaCl': -5.0, 'Diamond': -19.0, 'CsI3': -6.0, 'GeTe': -15.0}   # eV, from wannier90.win
N_e         = {'NaCl': 6,    'Diamond': 8,     'CsI3': 64,   'GeTe': 10}

CUT = 20.0   # only pairs with d <= 20 A

## Read COBICAR.lobster

File layout:
* line 1: comment
* line 2: `n_entries  n_spin  n_energy  E_min  E_max  E_F` (E_F is LOBSTER's Fermi level on the absolute DFT scale)
* then one label per entry: `Average`, then `No.1:Na1->Na1(19.31)`, ...
* then the data: each row is `E, COBI_avg, ICOBI_avg, COBI_1, ICOBI_1, COBI_2, ICOBI_2, ...`

Energies are relative to E_F. The ICOBI columns are cumulative, so they already include all states below the bottom of the grid.

In [3]:
def read_cobicar(path):
    with open(path) as f:
        lines = f.readlines()

    header = lines[1].split()
    n_energy = int(header[2])
    efermi = float(header[5])

    # labels like No.12:Cl2->Na1(2.84653)
    labels = []
    i = 2
    while not re.match(r'\s*-?\d', lines[i]):
        m = re.match(r'No\.(\d+):(\S+?)->(\S+?)\(([-\d.eE+]+)\)', lines[i])
        if m:
            labels.append([int(m.group(1)), m.group(2), m.group(3), float(m.group(4))])
        i += 1

    data = np.loadtxt(lines[i:i + n_energy])
    energy = data[:, 0]
    cobi  = data[:, 3::2]    # skip the Average columns (1,2)
    icobi = data[:, 4::2]
    return energy, cobi, icobi, labels, efermi

In [4]:
def read_icobilist(path):
    # ICOBILIST.lobster: COBI#  atomMU  atomNU  distance  T1 T2 T3  ICOBI(E_F)
    icobi = {}
    for line in open(path):
        p = line.split()
        if len(p) >= 8 and p[0].isdigit():
            icobi[int(p[0])] = float(p[7])
    return icobi

## Windowed ICOBI for every pair

* `ICOBI(E_start)`: interpolate the cumulative ICOBI at E_start. If E_start is below the grid (diamond), every state below the grid is inside the window, so subtract nothing.
* `ICOBI(E_F)`: interpolate at E = 0. The GeTe COBICAR grid stops below E_F (-22 to -10 eV), so for GeTe the value at E_F is taken from `ICOBILIST.lobster` of the same run.

In [5]:
def windowed_icobi(mat):
    energy, cobi, icobi, labels, efermi = read_cobicar(BASE + '/' + mat + '/LOBSTER/COBICAR.lobster')
    E_start = dis_win_min[mat] - efermi
    n_pairs = len(labels)

    # lower limit
    if E_start <= energy[0]:
        icobi_start = np.zeros(n_pairs)
    else:
        icobi_start = np.array([np.interp(E_start, energy, icobi[:, j]) for j in range(n_pairs)])

    # upper limit
    if energy[-1] >= 0:
        icobi_EF = np.array([np.interp(0.0, energy, icobi[:, j]) for j in range(n_pairs)])
    else:
        icobilist = read_icobilist(BASE + '/' + mat + '/LOBSTER/ICOBILIST.lobster')
        icobi_EF = np.array([icobilist[lab[0]] for lab in labels])

    df = pd.DataFrame(labels, columns=['index', 'atom1', 'atom2', 'distance'])
    df['ICOBI_EF'] = icobi_EF
    df['ICOBI_start'] = icobi_start
    df['ICOBI_W'] = icobi_EF - icobi_start

    print(f'{mat:8s} E_F = {efermi:8.4f}   E_start = {E_start:8.3f} eV   grid {energy[0]:.2f} to {energy[-1]:.2f} eV   {n_pairs} pairs')
    return df

In [6]:
pairs = {}
for mat in materials:
    pairs[mat] = windowed_icobi(mat)

pairs['NaCl'].sort_values('ICOBI_W', ascending=False).head()

NaCl     E_F =  -0.8220   E_start =   -4.178 eV   grid -10.03 to 10.03 eV   2034 pairs
Diamond  E_F =  10.0107   E_start =  -29.011 eV   grid -10.03 to 10.03 eV   8187 pairs
CsI3     E_F =   0.4334   E_start =   -6.433 eV   grid -10.03 to 10.03 eV   5268 pairs
GeTe     E_F =   5.8801   E_start =  -20.880 eV   grid -22.09 to -9.96 eV   1778 pairs


,index,atom1,atom2,distance,ICOBI_EF,ICOBI_start,ICOBI_W
1061,1062,Na1,Cl2,2.847184,0.08465,0.03479,0.04986
1070,1071,Na1,Cl2,2.847184,0.08465,0.03479,0.04986
982,983,Na1,Cl2,2.847184,0.08465,0.03479,0.04986
972,973,Na1,Cl2,2.847184,0.08464,0.03479,0.04985
1060,1061,Na1,Cl2,2.847184,0.08464,0.03479,0.04985


## Clean up the pair list

* element pair label, e.g. `Cl-Na` (sorted so Na-Cl and Cl-Na are the same)
* LOBSTER lists a pair between the **same** atom label (e.g. `C1->C1` at +T and -T) twice, so those rows get weight 1/2. Then summing the weights counts each pair once per primitive cell.
* keep 0 < d <= 20 A
* CsI3: only I-I pairs, because the Wannier model has I p orbitals only

In [7]:
def element(atom):
    return re.match(r'[A-Za-z]+', atom).group()

for mat in materials:
    df = pairs[mat]
    df['pair'] = ['-'.join(sorted([element(a), element(b)])) for a, b in zip(df['atom1'], df['atom2'])]
    df['weight'] = np.where(df['atom1'] == df['atom2'], 0.5, 1.0)
    df['bd2'] = df['weight'] * df['ICOBI_W'] * df['distance']**2

    keep = (df['distance'] > 1e-6) & (df['distance'] <= CUT)
    if mat == 'CsI3':
        keep = keep & (df['pair'] == 'I-I')
    pairs[mat] = df[keep].reset_index(drop=True)

## Total and Tr g

$\mathrm{Tr}\,g = \sum_\mathrm{pairs} b\,d^2 / 2N_e$

In [8]:
total = {}
trg = {}
for mat in materials:
    total[mat] = pairs[mat]['bd2'].sum()
    trg[mat] = total[mat] / (2 * N_e[mat])
    print(f'{mat:8s}  sum b d^2 (d <= 20 A) = {total[mat]:7.2f} A^2    Tr g (LOBSTER) = {trg[mat]:.3f} A^2   (N_e = {N_e[mat]})')

NaCl      sum b d^2 (d <= 20 A) =    3.72 A^2    Tr g (LOBSTER) = 0.310 A^2   (N_e = 6)
Diamond   sum b d^2 (d <= 20 A) =   12.20 A^2    Tr g (LOBSTER) = 0.762 A^2   (N_e = 8)
CsI3      sum b d^2 (d <= 20 A) =   84.44 A^2    Tr g (LOBSTER) = 0.660 A^2   (N_e = 64)
GeTe      sum b d^2 (d <= 20 A) =   57.27 A^2    Tr g (LOBSTER) = 2.864 A^2   (N_e = 10)


## Shell-resolved rows of Table S3

The listed shells are the five with the largest **Wannier** contribution (from the Wannier notebook), so they are typed in here. A pair belongs to a shell if it has the same elements and |d - d_shell| < 0.02 A.

In [9]:
table_shells = {
    'NaCl':    [('Cl-Na', 2.85), ('Cl-Na', 6.37), ('Cl-Cl', 4.03), ('Na-Na', 5.69), ('Na-Na', 4.03)],
    'Diamond': [('C-C', 1.55),   ('C-C', 3.89),   ('C-C', 5.05),   ('C-C', 2.96),   ('C-C', 6.38)],
    'CsI3':    [('I-I', 5.92),   ('I-I', 2.90),   ('I-I', 3.02),   ('I-I', 6.94),   ('I-I', 9.95)],
    'GeTe':    [('Ge-Te', 2.84), ('Ge-Te', 3.26), ('Ge-Te', 8.91), ('Ge-Te', 6.57), ('Ge-Ge', 6.08)],
}

In [10]:
rows = []
for mat in materials:
    df = pairs[mat]
    listed = 0.0
    for pair, d in table_shells[mat]:
        in_shell = (df['pair'] == pair) & (np.abs(df['distance'] - d) < 0.02)
        mult = df['weight'][in_shell].sum()
        bd2 = df['bd2'][in_shell].sum()
        b_L = (df['weight'] * df['ICOBI_W'])[in_shell].sum() / mult
        listed += bd2
        rows.append([mat, pair, d, mult, bd2, 100 * bd2 / total[mat], b_L])

    other = total[mat] - listed
    rows.append([mat, 'Other shells', None, None, other, 100 * other / total[mat], None])
    rows.append([mat, 'Total (d <= 20 A)', None, None, total[mat], 100.0, None])

table = pd.DataFrame(rows, columns=['material', 'atoms', 'd (A)', 'mult', 'sum b d^2 (A^2)', '%', 'b_L'])
table

,material,atoms,d (A),mult,sum b d^2 (A^2),%,b_L
0,NaCl,Cl-Na,2.85,6.0,2.424884,65.250965,0.049855
1,NaCl,Cl-Na,6.37,24.0,0.097277,2.617630,0.000100
2,NaCl,Cl-Cl,4.03,6.0,0.558373,15.025195,0.005740
3,NaCl,Na-Na,5.69,3.0,0.031129,0.837642,0.000320
4,NaCl,Na-Na,4.03,6.0,0.014592,0.392644,0.000150
5,NaCl,Other shells,NaN,NaN,0.589988,15.875924,NaN
6,NaCl,Total (d <= 20 A),NaN,NaN,3.716242,100.000000,NaN
7,Diamond,C-C,1.55,4.0,9.101172,74.603299,0.950640
8,Diamond,C-C,3.89,12.0,1.720782,14.105442,0.009460
9,Diamond,C-C,5.05,12.0,0.159307,1.305856,0.000520


In [11]:
pd.options.display.float_format = '{:.4f}'.format
for mat in materials:
    print(mat)
    print(table[table['material'] == mat].drop(columns='material').to_string(index=False))
    print(f'Tr g LOBSTER = {trg[mat]:.3f} A^2  (N_e = {N_e[mat]})')
    print()
pd.reset_option('display.float_format')

NaCl
            atoms  d (A)    mult  sum b d^2 (A^2)        %    b_L
            Cl-Na 2.8500  6.0000           2.4249  65.2510 0.0499
            Cl-Na 6.3700 24.0000           0.0973   2.6176 0.0001
            Cl-Cl 4.0300  6.0000           0.5584  15.0252 0.0057
            Na-Na 5.6900  3.0000           0.0311   0.8376 0.0003
            Na-Na 4.0300  6.0000           0.0146   0.3926 0.0001
     Other shells    NaN     NaN           0.5900  15.8759    NaN
Total (d <= 20 A)    NaN     NaN           3.7162 100.0000    NaN
Tr g LOBSTER = 0.310 A^2  (N_e = 6)

Diamond
            atoms  d (A)    mult  sum b d^2 (A^2)        %    b_L
              C-C 1.5500  4.0000           9.1012  74.6033 0.9506
              C-C 3.8900 12.0000           1.7208  14.1054 0.0095
              C-C 5.0500 12.0000           0.1593   1.3059 0.0005
              C-C 2.9600 12.0000           0.1959   1.6056 0.0019
              C-C 6.3800 24.0000           0.2246   1.8411 0.0002
     Other shells    NaN  

## Wannier bond indices (Wannier columns of Table S3)

The same quantity from the Wannier model: $\mathrm{Tr}\,g = \frac1{2N_e}\sum_{\alpha\beta}|\rho_{\alpha\beta}|^2|\mathbf R_{\alpha\beta}|^2$ with $\rho$ the density matrix of the occupied bands, and bond index $b = 4\sum|\rho_{\alpha\beta}|^2$ over the orbitals of the two atoms. The model is loaded as in `Fig2_minimal.ipynb` (§1–4) and the bond-resolved metric is computed as in `Fig4_minimal.ipynb` (§10.4). Hoppings with $|t| \ge 0.005$ eV.

### Create pythTB model from Wannier90, hopping array, orbital → atom, Hamiltonian on the k-mesh

In [ ]:
import itertools
from types import SimpleNamespace
from pymatgen.io.vasp.outputs import Outcar
from pythtb import *
ORB_MULT = {"s": 1, "p": 3, "d": 5, "f": 7, "sp": 2, "sp2": 3, "sp3": 4, "sp3d": 5, "sp3d2": 6}   # px, dxy, ... count 1

def win_block(txt, name):
    m = re.search(r"begin\s+" + name + r"\s*\n(.*?)\nend\s+" + name, txt, re.S | re.I)
    return [l.split("!")[0].split("#")[0].split() for l in m.group(1).splitlines()] if m else None


def load_wannier(material, min_hopping_norm, nk):
    M = SimpleNamespace(name=material)
    wann_dir = BASE + '/' + material

    #Create pythTB from wannier
    OUTCAR = Outcar(wann_dir + "/OUTCAR")
    mu = OUTCAR.efermi
    wannier_material = W90(wann_dir, r"wannier90")
    my_model = wannier_material.model(min_hopping_norm=min_hopping_norm)  # drop hoppings with |t| < min_hopping_norm

    #Create array with all hopping information
    M.lat = np.array(my_model.lattice.lat_vecs)                         # lattice vectors (rows), Angstrom
    M.orb_red = np.array(my_model.orb_vecs)
    hoppings = [(hop['amplitude'], hop['from_orbital'], hop['to_orbital'], hop.get('lattice_vector', [0, 0, 0]))
                for hop in my_model.hoppings]
    M.orb_pos = M.orb_red @ M.lat                                       # Wannier centres, cartesian
    M.norb = len(M.orb_pos)
    M.onsite = np.real(np.array(my_model._site_energies, dtype=complex))
    hop_from, hop_to, hop_R, hop_t = [], [], [], []
    for t, i, j, j_image in hoppings:
        j_image = np.array(j_image, dtype=int)
        hop_from += [i, j];  hop_to += [j, i]                           # i -> j+R  and its partner  j -> i-R
        hop_R += [j_image, -j_image]
        hop_t += [t, np.conj(t)]
    M.hop_from = np.array(hop_from); M.hop_to = np.array(hop_to)
    M.hop_R = np.array(hop_R); M.hop_t = np.array(hop_t, dtype=complex)
    M.hop_disp = (M.hop_R + M.orb_red[M.hop_to] - M.orb_red[M.hop_from]) @ M.lat   # d = R + tau_j - tau_i

    #Order of pythTB orbitals: which atom each orbital belongs to
    win = open(wann_dir + "/wannier90.win").read()
    rows = [r for r in (win_block(win, "atoms_cart") or win_block(win, "atoms_frac")) if r]
    M.species = [r[0] for r in rows]
    M.atom_pos = np.array([r[1:4] for r in rows], float)
    if win_block(win, "atoms_cart") is None:
        M.atom_pos = M.atom_pos @ M.lat                                 # atoms_frac -> cartesian
    orb_atom = []
    for r in win_block(win, "projections"):
        if not r: continue
        site, orbs = r[0].split(":")[:2]                                # e.g. "Ge:s,p"
        n_orb = sum(ORB_MULT.get(o.strip().lower(), 1) for o in re.split("[,;]", orbs))
        orb_atom += [a for a, s in enumerate(M.species) if s == site for _ in range(n_orb)]
    M.orb_atom = np.array(orb_atom)
    M.orb_cell = np.round((M.orb_pos - M.atom_pos[M.orb_atom]) @ np.linalg.inv(M.lat)).astype(int)

    #Hamiltonian on the k-mesh
    M.nk = nk
    M.recip = 2 * np.pi * np.linalg.inv(M.lat).T
    g = (np.arange(nk) + 0.5) / nk
    k_frac = np.array(np.meshgrid(g, g, g, indexing="ij")).reshape(3, -1).T
    M.kSpan = k_frac @ M.recip                                          # cartesian k, nk^3 points
    M.H_k = ham_func_chunked(M, M.kSpan)
    M.E_k, M.U_k = np.linalg.eigh(M.H_k)                                # U_k[k, orbital, band]
    M.occ = int(np.bincount((M.E_k < mu).sum(1)).argmax())              # occupied bands
    print(material, ':', M.norb, "orbitals,", len(M.hop_t), "directed hoppings,  nk %d,  occ %d" % (nk, M.occ))
    return M


def ham_func(M, kpts):
    #H(k) for an array of cartesian k-points (shape nk x 3)
    phase = np.exp(1j * kpts @ M.hop_disp.T) * M.hop_t                  # (nk, nhop): t e^{ik.d}
    to_matrix = np.zeros((len(M.hop_t), M.norb * M.norb))               # hops -> matrix element (i, j)
    to_matrix[np.arange(len(M.hop_t)), M.hop_from * M.norb + M.hop_to] = 1
    return (phase @ to_matrix).reshape(-1, M.norb, M.norb) + np.diag(M.onsite)

def ham_func_chunked(M, kpts, chunk=2000):
    return np.concatenate([ham_func(M, kpts[s:s + chunk]) for s in range(0, len(kpts), chunk)])

### Bond-resolved quantum metric

`rho[o, o', R]` = ⟨o, cell R | P | o', home cell⟩ from a Fourier transform over the mesh. Every bond $o'$ (home cell) → $o$ (cell R) adds $|\rho|^2|\mathbf R|^2/2N_\mathrm{occ}$, with $\mathbf R$ between the orbital centres. Bonds are grouped into shells by the two atoms and their distance; bonds within one atom are kept in the total only.

In [ ]:
def density_matrix(M, bands, gt, cells):
    #rho[o, o', cell] = <o, cell R | U g U^dagger | o', home cell>  (Fourier transform over the mesh)
    orb_phase = np.exp(1j * M.kSpan @ M.orb_pos.T)                      # e^{ik.tau_o}
    P = orb_phase[:, :, None] * orb_phase.conj()[:, None, :]            # e^{ik(tau_o - tau_o')}
    Ub = M.U_k[:, :, bands]
    rho_k = np.einsum("kai,ki,kbi->kab", Ub, gt, Ub.conj()) * P         # U g U^dagger
    F = np.fft.ifftn(rho_k.reshape(M.nk, M.nk, M.nk, M.norb, M.norb), axes=(0, 1, 2))   # (1/N) sum_k e^{ik.R}
    shift_phase = np.exp(1j * np.pi * cells.sum(1) / M.nk)              # half-step mesh shift
    return (F[tuple((cells % M.nk).T)] * shift_phase[:, None, None]).transpose(1, 2, 0)


def bond_resolved(M):
    #all cells the mesh resolves without aliasing
    n = (M.nk - 1) // 2
    cells = np.array(list(itertools.product(range(-n, n + 1), repeat=3)))
    rho = density_matrix(M, slice(0, M.occ), np.ones((len(M.kSpan), M.occ)), cells)

    #every bond o' (home cell) -> o (cell R): |rho|^2 |R|^2 / (2 occ)
    RL = cells @ M.lat
    disp = M.orb_pos[:, None, None, :] + RL[None, None] - M.orb_pos[None, :, None, :]       # (o, o', cell, 3)
    trg = abs(rho) ** 2 * (disp ** 2).sum(-1) / (2 * M.occ)
    M.trg_bonds = trg.sum()

    #the atoms of the two orbitals: atom of o in cell R + its own cell, atom of o' in its own cell
    atom_of = M.atom_pos[M.orb_atom] + M.orb_cell @ M.lat
    atom_dist = np.linalg.norm(atom_of[:, None, None, :] + RL[None, None] - atom_of[None, :, None, :], axis=-1)
    o, o2, c = np.nonzero((atom_dist > 0.05) & (atom_dist <= CUT))
    bonds = pd.DataFrame({'atom_from': M.orb_atom[o2], 'atom_to': M.orb_atom[o],
                          'R1': cells[c, 0] + M.orb_cell[o, 0] - M.orb_cell[o2, 0],
                          'R2': cells[c, 1] + M.orb_cell[o, 1] - M.orb_cell[o2, 1],
                          'R3': cells[c, 2] + M.orb_cell[o, 2] - M.orb_cell[o2, 2],
                          'distance': atom_dist[o, o2, c], 'trg': trg[o, o2, c]})
    bonds['pair'] = ['-'.join(sorted((M.species[a], M.species[b]))) for a, b in zip(bonds.atom_from, bonds.atom_to)]
    bonds['shell'] = bonds['distance'].round(2)

    #bonds of the same atoms and length -> one shell (each bond is listed once from each end)
    shells = bonds.groupby(['pair', 'shell']).agg(trg=('trg', 'sum')).reset_index()
    atom_pairs = bonds.drop_duplicates(['pair', 'shell', 'atom_from', 'atom_to', 'R1', 'R2', 'R3'])
    shells['mult'] = atom_pairs.groupby(['pair', 'shell']).size().values / 2
    shells['bd2'] = 4 * M.occ * shells['trg']                           # sum of b d^2 = 2 N_e x (share of Tr g)
    shells['bond_index'] = shells['bd2'] / shells['mult'] / shells['shell'] ** 2
    M.shells = shells.sort_values('bd2', ascending=False).reset_index(drop=True)
    print("   bond-resolved Tr g = %.4f A^2" % M.trg_bonds)
    return M

In [ ]:
wannier_settings = {'NaCl': 30, 'Diamond': 30, 'CsI3': 15, 'GeTe': 30}   # nk
W = {mat: bond_resolved(load_wannier(mat, 0.005, wannier_settings[mat])) for mat in materials}

### Table S3: the five shells with the largest Wannier contribution, Wannier next to LOBSTER

In [ ]:
rows = []
for mat in materials:
    ws = W[mat].shells
    totW = ws['bd2'].sum()
    df = pairs[mat]; listedW = listedL = 0.0
    for _, s in ws.head(5).iterrows():
        pair, d = s['pair'], s['shell']
        in_shell = (df['pair'] == pair) & (np.abs(df['distance'] - d) < 0.02)
        multL = df['weight'][in_shell].sum(); bd2L = df['bd2'][in_shell].sum()
        bL = (df['weight'] * df['ICOBI_W'])[in_shell].sum() / multL if multL else np.nan
        listedW += s['bd2']; listedL += bd2L
        rows.append([mat, pair, d, s['mult'], s['bd2'], 100 * s['bd2'] / totW, bd2L, 100 * bd2L / total[mat],
                     s['bond_index'], bL])
    rows.append([mat, 'Other shells', None, None, totW - listedW, 100 * (totW - listedW) / totW,
                 total[mat] - listedL, 100 * (total[mat] - listedL) / total[mat], None, None])
    rows.append([mat, 'Total (d <= 20 A)', None, None, totW, 100.0, total[mat], 100.0, None, None])
table_S3 = pd.DataFrame(rows, columns=['material', 'atoms', 'd (A)', 'mult', 'Wannier sum b d^2', 'Wannier %',
                                       'LOBSTER sum b d^2', 'LOBSTER %', 'b_W', 'b_L'])
pd.options.display.float_format = '{:.4f}'.format
for mat in materials:
    print(mat)
    print(table_S3[table_S3['material'] == mat].drop(columns='material').to_string(index=False))
    print(f"Tr g: Wannier {W[mat].trg_bonds:.3f} A^2, LOBSTER {trg[mat]:.3f} A^2  (N_e = {N_e[mat]})\n")
pd.reset_option('display.float_format')